If you feed these two sentences into a standard network (without positional info), the **meaning is lost without a way to encode positional encoding**:

* *"AI writes code"*
* *"code writes AI"*

### Positional Encoding (Vaswani et al., 2017)

Für ein Token an Position `pos` und eine Dimension `i` im Positionsvektor `d` gilt:

$$
\text{PE}(pos, 2i) = \sin\Big(\frac{pos}{10000^{2i/d}}\Big)
$$

$$
\text{PE}(pos, 2i+1) = \cos\Big(\frac{pos}{10000^{2i/d}}\Big)
$$

- `pos` = Position des Tokens im Satz (0, 1, 2, …)  
- `i` = Index der Dimension im Positionsvektor (0, 1, 2, …, d/2)  
- `d` = Gesamtdimension des Embeddings  

**Hinweis:** Gerade Indizes → Sinus, ungerade Indizes → Kosinus.  
Die Basis `10000` sorgt dafür, dass hohe Dimensionen langsamer variieren (slow) und niedrige Dimensionen schneller (fast).


We must inject "order" before the data hits the first layer. We add a unique "ID badge" (a vector) to every word based on its position in the sentence (1st, 2nd, 3rd...).

text
      "I"        "love"       "AI"       <-- Word Embeddings (Semantic meaning)
       +           +           +
    [Pos 0]     [Pos 1]     [Pos 2]      <-- Positional Encodings (Order info)
       |           |           |
       v           v           v
    [Vec 0]     [Vec 1]     [Vec 2]      <-- Transformer Input
    


#### 1. "AI" (Position 0)
**Index 0 (Even):** $\sin(0 / 1) = \sin(0) = \mathbf{0.00}$
**Index 1 (Odd):** $\cos(0 / 1) = \cos(0) = \mathbf{1.00}$
**Index 2 (Even):** $\sin(0 / 464) = \sin(0) = \mathbf{0.00}$
**Vector:** `[0.00, 1.00, 0.00]`
* Intuition: "Start."*

#### 2. "writes" (Position 1)
**Index 0 (Fast):** $\sin(1 / 1) = \sin(1 \text{ rad}) \approx \mathbf{0.84}$
    * *(Big jump from 0.00)*
**Index 1 (Fast):** $\cos(1 / 1) = \cos(1 \text{ rad}) \approx \mathbf{0.54}$
    * *(Big jump from 1.00)*
**Index 2 (Slow):** $\sin(1 / 464) \approx \mathbf{0.002}$
    * *(Tiny shift from 0.00)*
**Vector:** `[0.84, 0.54, 0.002]`
* Intuition: "Near #1."*

#### 3. "Code" (Position 2)
**Index 0 (Fast):** $\sin(2 / 1) = \sin(2 \text{ rad}) \approx \mathbf{0.91}$
    * *(Chaotic jump)*
**Index 1 (Fast):** $\cos(2 / 1) = \cos(2 \text{ rad}) \approx \mathbf{-0.42}$
    * *(Chaotic jump)*
**Index 2 (Slow):** $\sin(2 / 464) \approx \mathbf{0.004}$
    * *(Tiny steady increment)*
**Vector:** `[0.91, -0.42, 0.004]`
* Intuition: "Near #2, but we are still in the early section."*

In [1]:
import torch
import math

def get_positional_encoding(max_seq_len, d_model):
    pe = torch.zeros(max_seq_len, d_model)    
    position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)    
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )   
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    
    return pe

d_model = 4
max_len = 3
pe_matrix = get_positional_encoding(max_len, d_model)

print("Pos 0 AI :", pe_matrix[0])
print("Pos 1 writes :", pe_matrix[1])
print("Pos 2 code :", pe_matrix[2])



Pos 0 AI : tensor([0., 1., 0., 1.])
Pos 1 writes : tensor([0.8415, 0.5403, 0.0100, 0.9999])
Pos 2 code : tensor([ 0.9093, -0.4161,  0.0200,  0.9998])


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SelfAttention(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()
        self.W_q = nn.Linear(d_model, head_size, bias=False)
        self.W_k = nn.Linear(d_model, head_size, bias=False)
        self.W_v = nn.Linear(d_model, head_size, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.W_q(x) 
        k = self.W_k(x)
        v = self.W_v(x)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(k.size(-1))
        weights = F.softmax(scores, dim=-1)

        # 5. Aggregate Information (Weights @ V)
        # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        context = weights @ v

        return context

d_model = 2
x = torch.tensor([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]) 
sa = SelfAttention(d_model=2, head_size=2)

with torch.no_grad():
    sa.W_q.weight.copy_(torch.eye(2))
    sa.W_k.weight.copy_(torch.eye(2))
    sa.W_v.weight.copy_(torch.eye(2))

output = sa(x)
print("Context Vector for 'love' (Index 1):")
print(output[0, 1])

Context Vector for 'love' (Index 1):
tensor([0.5989, 0.8022], grad_fn=<SelectBackward0>)


Self-Attention

**Database/Search Engine** analogy:

* **Query ($Q$):** *What am I looking for?* (e.g., "love" looking for a subject or object).
* **Key ($K$):** *What do I describe?* (e.g., "AI" advertising itself as a noun).
* **Value ($V$):** *What information do I hold?* (e.g., The semantic meaning of "AI").

**The Core Idea:**
Every token asks a question ($Q$). Every other token answers with its relevance ($K$). If the relevance is high, the first token absorbs the information ($V$) of the second.




In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

x = torch.tensor([[
    [0.00, 1.00, 0.00, 1.00],  # AI
    [0.84, 0.54, 0.01, 1.00],  # writes
    [0.91, -0.42, 0.02, 1.00]  # code
]])

class SelfAttention(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()
        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        
        scores = q @ k.transpose(-2, -1)        
        print("Raw Scores for 'writes':", scores[0, 1].detach().numpy())        
        att_weights = F.softmax(scores / math.sqrt(C), dim=-1)
       
        out = att_weights @ v
        return out

d_model = 4
sa = SelfAttention(d_model, d_model)
with torch.no_grad():
    sa.key.weight.copy_(torch.eye(d_model))
    sa.query.weight.copy_(torch.eye(d_model))
    sa.value.weight.copy_(torch.eye(d_model))

output = sa(x)
print("Output Vector for 'writes':", output[0, 1].detach().numpy())

Raw Scores for 'writes': [1.54      1.9973    1.5378001]
Output Vector for 'writes': [0.6034756  0.3867522  0.00999662 1.        ]


Let's assume our word embeddings are zeros, so the **Input ($X$)** is exactly those Positional Encoding vectors we calculated.

**The Input Vectors:**
* **"AI" (Pos 0):** `[0.0, 1.0, 0.0, 1.0]`
* **"writes" (Pos 1):** `[0.84, 0.54, 0.01, 1.0]`
* **"code" (Pos 2):** `[0.91, -0.42, 0.02, 1.0]`


We stack these vectors to form our Input Matrix $X$.
* **Rows:** 3 ("AI", "writes", "code")
* **Columns:** 4 ($d_{model}$)

$$
X =
\begin{bmatrix}
0.0 & 1.0 & 0.0 & 1.0 \\
0.84 & 0.54 & 0.01 & 1.0 \\
0.91 & -0.42 & 0.02 & 1.0
\end{bmatrix}
$$

Lets define our **Weights** ($W_Q, W_K, W_V$).
In PyTorch, weights are stored as `[out_features, in_features]`. 

To multiply them with our input $X$ `[batch, in_features]`, we must **transpose** the weights.

For this tutorial, we assume **Identity Matrices** for simplicity:

$$
W_Q = W_K = W_V =
\begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}
$$


We create our Query, Key, and Value matrices by projecting the input $X$ through the weight matrices.

$$Q = X \cdot W_Q^T = \begin{bmatrix} 0.0 & 1.0 & 0.0 & 1.0 \\ 0.84 & 0.54 & 0.01 & 1.0 \\ 0.91 & -0.42 & 0.02 & 1.0 \end{bmatrix}$$

$$K = X \cdot W_K^T = \begin{bmatrix} 0.0 & 1.0 & 0.0 & 1.0 \\ 0.84 & 0.54 & 0.01 & 1.0 \\ 0.91 & -0.42 & 0.02 & 1.0 \end{bmatrix}$$

$$V = X \cdot W_V^T = \begin{bmatrix} 0.0 & 1.0 & 0.0 & 1.0 \\ 0.84 & 0.54 & 0.01 & 1.0 \\ 0.91 & -0.42 & 0.02 & 1.0 \end{bmatrix}$$


We calculate similarity by multiplying $Q$ by the **Transpose of K**.

$$
\text{Scores} = Q \cdot K^T =
\begin{bmatrix}
0.0 & 1.0 & 0.0 & 1.0 \\
0.84 & 0.54 & 0.01 & 1.0 \\
0.91 & -0.42 & 0.02 & 1.0
\end{bmatrix}
\times
\begin{bmatrix}
0.0 & 0.84 & 0.91 \\
1.0 & 0.54 & -0.42 \\
0.0 & 0.01 & 0.02 \\
1.0 & 1.0 & 1.0
\end{bmatrix}
$$

Here I am just giving scores for **Row 2 ("writes")** vs **Row 1 ("AI")**:
$$(0.84 \times 0) + (0.54 \times 1) + (0.01 \times 0) + (1 \times 1)$$
$$= 0 + 0.54 + 0 + 1 = \mathbf{1.54}$$


$$
\text{Scores} \approx
\begin{bmatrix}
2.00 & 1.54 & 0.58 \\
1.54 & 1.99 & 1.54 \\
0.58 & 1.54 & 2.00
\end{bmatrix}
$$

We apply Softmax to Row 2 to turn raw scores into probabilities.

$$
\text{Attn}_{writes} = \text{softmax}([1.54, 1.99, 1.54]) \approx \mathbf{[0.28, 0.44, 0.28]}
$$

We multiply the Attention Weights by the Value Matrix $V$ (which is $X \cdot W_V^T$).

$$
Z_{writes} = [0.28, 0.44, 0.28] \times
\begin{bmatrix}
0.0 & 1.0 & 0.0 & 1.0 \\
0.84 & 0.54 & 0.01 & 1.0 \\
0.91 & -0.42 & 0.02 & 1.0
\end{bmatrix}
$$

$$(0.28 \times 0.0) + (0.44 \times 0.84) + (0.28 \times 0.91)$$
$$= 0 + 0.37 + 0.25 = \mathbf{0.62}$$

***Multi Head Attention***

* **Head 1 (Grammar Lane):** Focuses on syntax. It might want "writes" to look *only* at "AI" (Subject-Verb agreement).
* **Head 2 (Context Lane):** Focuses on meaning. It might want "writes" to look *only* at "code" (Verb-Object relationship).

**Multi-Head Attention** simply runs several Self-Attention operations in parallel. Each head has its own unique weight matrices ($W_Q, W_K, W_V$), allowing it to specialize in learning different relationships.

$$
\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O
$$
where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

---

Our embedding size ($d_{model}$) is 4. We want 2 Heads.
Instead of one big matrix multiplication (size 4), we slice the vector into two smaller chunks (size 2 each).



```text
Input Vector (size 4): [x1, x2, x3, x4]
       |
       v
    Split!
   /      \
Head 1    Head 2
[x1, x2]  [x3, x4]  <-- Each head processes a smaller vector independently.
   |         |
Attention  Attention
   |         |
[z1, z2]  [z3, z4]  <-- Results
   \      /
    Concat
       v
[z1, z2, z3, z4]    <-- Final Output

***KV CACHING***

Imagine you are reading a book.
* **Word 1:** You read "The".
* **Word 2:** To read "dog", you start over: "The... dog".
* **Word 3:** To read "barks", you start over: "The... dog... barks".

This is how a Transformer works by default during inference. It processes the **entire history** every time it generates one new word.

### Visualizing the BS 

Without caching, generating the 3rd word "code" looks like this:

```text
Step 1: Generate "writes" (Input: "AI")
[Process "AI"] ---> Output "writes"

Step 2: Generate "code" (Input: "AI writes")
[Process "AI"]      <-- WASTED MATH (We already did this!)
[Process "writes"]  <-- New Math
       |
    Output "code"

We simply **save** the Key ($K$) and Value ($V$) vectors of past tokens in GPU memory. We never calculate them again.

Let's generate the token **"code"** given the context **"AI writes"**.
* **Cache:** We already possess $K$ and $V$ for "AI" and "writes" in memory.
* **New Input:** We only process the single token **"code"**.


**1. Calculate Q, K, V for NEW token only**
We only run the heavy weights ($W_Q, W_K, W_V$) on "code".

$$Q_{new} = [1 \times 4] \quad (\text{"code"})$$
$$K_{new} = [1 \times 4] \quad (\text{"code"})$$
$$V_{new} = [1 \times 4] \quad (\text{"code"})$$

**2. Retrieve and Stack (The Cache)**
We grab the old keys from memory and stack the new one on top.

```text
       K_Cache (Memory)           K_new
     +-----------------+       +---------+
(AI) | 0.0  1.0  0.0  1|       |         |
(wr) | 0.8  0.5  0.0  1|   +   | 0.9 ... |  (code)
     +-----------------+       +---------+

Q (code)           K_Stack_Transposed (AI, writes, code)
     [1 x 4]      x             [4 x 3]

                          (AI)   (wr)   (code)
                       +------+------+------+
                       | 0.0  | 0.8  | 0.9  |
                       | 1.0  | 0.5  | ...  |
                       | ...  | ...  | ...  |
                       | ...  | ...  | ...  |
                       +------+------+------+
                                  |
                                  v
                           Scores [1 x 3]
                     (How much "code" cares about history)

In [ ]:
from typing import Tuple, Optional

import torch
import triton
import triton.language as tl
from triton import Config


@triton.jit
def act_quant_kernel(x_ptr, y_ptr, s_ptr, BLOCK_SIZE: tl.constexpr, scale_fmt: tl.constexpr):
    """
    Quantizes the input tensor `x_ptr` and stores the result in `y_ptr` and the scaling factor in `s_ptr`.

    Args:
        x_ptr (triton.Pointer): Pointer to the input tensor.
        y_ptr (triton.Pointer): Pointer to the output tensor where quantized values will be stored.
        s_ptr (triton.Pointer): Pointer to the output tensor where scaling factors will be stored.
        BLOCK_SIZE (tl.constexpr): The size of the block to be processed by each program instance.

    Returns:
        None
    """
    pid = tl.program_id(axis=0)
    offs = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    x = tl.load(x_ptr + offs).to(tl.float32)
    amax = tl.max(tl.abs(x)) 
    amax = tl.maximum(amax, 1e-4) e3
    s = amax / 448.
    if scale_fmt == "ue8m0":
        exp = tl.math.ceil(tl.math.log2(s))
        s = tl.math.exp2(exp)
    y = x / s
    y = y.to(y_ptr.dtype.element_ty)
    tl.store(y_ptr + offs, y)
    tl.store(s_ptr + pid, s)


def act_quant(x: torch.Tensor, block_size: int = 128, scale_fmt: Optional[str] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Quantizes the input tensor `x` using block-wise quantization.

    Args:
        x (torch.Tensor): The input tensor to be quantized. Must be contiguous and its last dimension size must be divisible by `block_size`.
        block_size (int, optional): The size of the blocks to be used for quantization. Default is 128.
        scale_fmt (Optional[str], optional): The format of the scale. Default is None.
    Returns:
        Tuple[torch.Tensor, torch.Tensor]: A tuple containing:
            - The quantized tensor with dtype `torch.float8_e4m3fn`.
            - A tensor of scaling factors with dtype `torch.float32`.
    """
    assert x.is_contiguous(), 'Input tensor must be contiguous'
    assert x.size(-1) % block_size == 0, f'Last dimension size must be divisible by block_size (block_size={block_size})'
    y = torch.empty_like(x, dtype=torch.float8_e4m3fn)
    s = x.new_empty(*x.size()[:-1], x.size(-1) // block_size, dtype=torch.float32)
    grid = lambda meta: (triton.cdiv(x.numel(), meta['BLOCK_SIZE']), )
    act_quant_kernel[grid](x, y, s, BLOCK_SIZE=block_size, scale_fmt=scale_fmt)
    return y, s


@triton.jit
def weight_dequant_kernel(x_ptr, s_ptr, y_ptr, M, N, BLOCK_SIZE: tl.constexpr):
    """
    Dequantizes weights using the provided scaling factors and stores the result.

    Args:
        x_ptr (tl.pointer): Pointer to the quantized weights.
        s_ptr (tl.pointer): Pointer to the scaling factors.
        y_ptr (tl.pointer): Pointer to the output buffer for dequantized weights.
        M (int): Number of rows in the weight matrix.
        N (int): Number of columns in the weight matrix.
        BLOCK_SIZE (tl.constexpr): Size of the block for tiling.

    Returns:
        None
    """
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)
    n = tl.cdiv(N, BLOCK_SIZE)
    offs_m = pid_m * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    offs_n = pid_n * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    offs = offs_m[:, None] * N + offs_n[None, :]
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    x = tl.load(x_ptr + offs, mask=mask).to(tl.float32)
    s = tl.load(s_ptr + pid_m * n + pid_n)
    y = x * s
    tl.store(y_ptr + offs, y, mask=mask)


def weight_dequant(x: torch.Tensor, s: torch.Tensor, block_size: int = 128) -> torch.Tensor:
    """
    Dequantizes the given weight tensor using the provided scale tensor.

    Args:
        x (torch.Tensor): The quantized weight tensor of shape (M, N).
        s (torch.Tensor): The scale tensor of shape (M//block_size, N//block_size).
        block_size (int, optional): The block size to use for dequantization. Defaults to 128.

    Returns:
        torch.Tensor: The dequantized weight tensor of the same shape as `x`.

    Raises:
        AssertionError: If `x` or `s` are not contiguous or if their dimensions are not 2.
    """
    assert x.is_contiguous() and s.is_contiguous(), 'Input tensors must be contiguous'
    assert x.dim() == 2 and s.dim() == 2, 'Input tensors must have 2 dimensions'
    M, N = x.size()
    y = torch.empty_like(x, dtype=torch.get_default_dtype())
    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE']), triton.cdiv(N, meta['BLOCK_SIZE']))
    weight_dequant_kernel[grid](x, s, y, M, N, BLOCK_SIZE=block_size)
    return y


fp8_gemm_configs = [
    Config({'BLOCK_SIZE_M': block_m, 'BLOCK_SIZE_N': block_n, 'BLOCK_SIZE_K': 128}, num_stages=num_stages, num_warps=8)
    for block_m in [16, 32, 64] for block_n in [32, 64, 128] for num_stages in [3, 4, 5, 6]
]

@triton.autotune(configs=fp8_gemm_configs, key=['N', 'K'])
@triton.jit
def fp8_gemm_kernel(a_ptr, b_ptr, c_ptr,
                    a_s_ptr, b_s_ptr,
                    M, N: tl.constexpr, K: tl.constexpr,
                    BLOCK_SIZE_M: tl.constexpr,
                    BLOCK_SIZE_N: tl.constexpr,
                    BLOCK_SIZE_K: tl.constexpr):
    """
    Performs a matrix multiplication operation on FP8 matrices with scaling factors.

    Args:
        a_ptr (tl.tensor): Pointer to the first input matrix A.
        b_ptr (tl.tensor): Pointer to the second input matrix B.
        c_ptr (tl.tensor): Pointer to the output matrix C.
        a_s_ptr (tl.tensor): Pointer to the scaling factors for matrix A.
        b_s_ptr (tl.tensor): Pointer to the scaling factors for matrix B.
        M (int): Number of rows in matrix A and C.
        N (tl.constexpr): Number of columns in matrix B and C.
        K (tl.constexpr): Number of columns in matrix A and rows in matrix B.
        BLOCK_SIZE_M (tl.constexpr): Block size for the M dimension.
        BLOCK_SIZE_N (tl.constexpr): Block size for the N dimension.
        BLOCK_SIZE_K (tl.constexpr): Block size for the K dimension.

    Returns:
        None
    """
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)
    k = tl.cdiv(K, BLOCK_SIZE_K)
    offs_m = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_n = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + offs_m[:, None] * K + offs_k[None, :]
    b_ptrs = b_ptr + offs_n[None, :] * K + offs_k[:, None]
    a_s_ptrs = a_s_ptr + offs_m * k
    b_s_ptrs = b_s_ptr + (offs_n // BLOCK_SIZE_K) * k

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for i in range(k):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - i * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - i * BLOCK_SIZE_K, other=0.0)
        a_s = tl.load(a_s_ptrs)
        b_s = tl.load(b_s_ptrs)
        accumulator += tl.dot(a, b) * a_s[:, None] * b_s[None, :]
        a_ptrs += BLOCK_SIZE_K
        b_ptrs += BLOCK_SIZE_K
        a_s_ptrs += 1
        b_s_ptrs += 1
    c = accumulator.to(c_ptr.dtype.element_ty)
    offs_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + offs_m[:, None] * N + offs_n[None, :]
    mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(c_ptrs, c, mask=mask)


def fp8_gemm(a: torch.Tensor, a_s: torch.Tensor, b: torch.Tensor, b_s: torch.Tensor):
    """
    Perform a matrix multiplication using FP8 precision.

    Args:
        a (torch.Tensor): The first input matrix, must be contiguous.
        a_s (torch.Tensor): The scaling factor for the first input matrix, must be contiguous.
        b (torch.Tensor): The second input matrix, must be contiguous.
        b_s (torch.Tensor): The scaling factor for the second input matrix, must be contiguous.

    Returns:
        torch.Tensor: The result of the matrix multiplication.
    """
    assert a.is_contiguous() and b.is_contiguous(), 'Input tensors must be contiguous'
    assert a_s.is_contiguous() and b_s.is_contiguous(), 'Scaling factor tensors must be contiguous'
    K = a.size(-1)
    M = a.numel() // K
    N = b.size(0)
    c = a.new_empty(*a.size()[:-1], N, dtype=torch.get_default_dtype())
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']), triton.cdiv(N, META['BLOCK_SIZE_N']))
    fp8_gemm_kernel[grid](a, b, c, a_s, b_s, M, N, K)
    return c

In [11]:
import math
from dataclasses import dataclass
from typing import Tuple, Optional, Literal

import torch
from torch import nn
import torch.nn.functional as F
import torch.distributed as dist



world_size = 1
rank = 0
block_size = 128
gemm_impl: Literal["bf16", "fp8"] = "bf16"
attn_impl: Literal["naive", "absorb"] = "absorb"

@dataclass
class ModelArgs:
    """
    Data class for defining model arguments and hyperparameters.

    Attributes:
        max_batch_size (int): Maximum batch size.
        max_seq_len (int): Maximum sequence length.
        dtype (Literal["bf16", "fp8"]): Data type for computations.
        scale_fmt (Optional[str]): Format for quantization scale.
        vocab_size (int): Vocabulary size.
        dim (int): Model dimension.
        inter_dim (int): Intermediate dimension for MLP layers.
        moe_inter_dim (int): Intermediate dimension for MoE layers.
        n_layers (int): Number of transformer layers.
        n_dense_layers (int): Number of dense layers in the model.
        n_heads (int): Number of attention heads.
        n_routed_experts (int): Number of routed experts for MoE layers.
        n_shared_experts (int): Number of shared experts for MoE layers.
        n_activated_experts (int): Number of activated experts in MoE layers.
        n_expert_groups (int): Number of expert groups.
        n_limited_groups (int): Number of limited groups for MoE routing.
        score_func (Literal["softmax", "sigmoid"]): Scoring function for MoE routing.
        route_scale (float): Scaling factor for routing scores.
        q_lora_rank (int): LoRA rank for query projections.
        kv_lora_rank (int): LoRA rank for key-value projections.
        qk_nope_head_dim (int): Dimension for query-key projections without positional embeddings.
        qk_rope_head_dim (int): Dimension for query-key projections with rotary embeddings.
        v_head_dim (int): Dimension for value projections.
        original_seq_len (int): Original sequence length.
        rope_theta (float): Base for rotary positional encoding.
        rope_factor (float): Scaling factor for extended sequence lengths.
        beta_fast (int): Fast beta correction factor.
        beta_slow (int): Slow beta correction factor.
        mscale (float): Scaling factor for extended attention.
    """
    max_batch_size: int = 8
    max_seq_len: int = 4096 * 4
    dtype: Literal["bf16", "fp8"] = "bf16"
    scale_fmt: Optional[str] = None
    vocab_size: int = 102400
    dim: int = 2048
    inter_dim: int = 10944
    moe_inter_dim: int = 1408
    n_layers: int = 27
    n_dense_layers: int = 1
    n_heads: int = 16
    # moe
    n_routed_experts: int = 64
    n_shared_experts: int = 2
    n_activated_experts: int = 6
    n_expert_groups: int = 1
    n_limited_groups: int = 1
    score_func: Literal["softmax", "sigmoid"] = "softmax"
    route_scale: float = 1.
    # mla
    q_lora_rank: int = 0
    kv_lora_rank: int = 512
    qk_nope_head_dim: int = 128
    qk_rope_head_dim: int = 64
    v_head_dim: int = 128
    # yarn
    original_seq_len: int = 4096
    rope_theta: float = 10000.0
    rope_factor: float = 40
    beta_fast: int = 32
    beta_slow: int = 1
    mscale: float = 1.



class MLA(nn.Module):
    """
    Multi-Head Latent Attention (MLA) Layer.

    Attributes:
        dim (int): Dimensionality of the input features.
        n_heads (int): Number of attention heads.
        n_local_heads (int): Number of local attention heads for distributed systems.
        q_lora_rank (int): Rank for low-rank query projection.
        kv_lora_rank (int): Rank for low-rank key/value projection.
        qk_nope_head_dim (int): Dimensionality of non-positional query/key projections.
        qk_rope_head_dim (int): Dimensionality of rotary-positional query/key projections.
        qk_head_dim (int): Total dimensionality of query/key projections.
        v_head_dim (int): Dimensionality of value projections.
        softmax_scale (float): Scaling factor for softmax in attention computation.
    """
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.dim = args.dim
        self.n_heads = args.n_heads
        self.n_local_heads = args.n_heads // world_size
        self.q_lora_rank = args.q_lora_rank
        self.kv_lora_rank = args.kv_lora_rank
        self.qk_nope_head_dim = args.qk_nope_head_dim
        self.qk_rope_head_dim = args.qk_rope_head_dim
        self.qk_head_dim = args.qk_nope_head_dim + args.qk_rope_head_dim
        self.v_head_dim = args.v_head_dim

        if self.q_lora_rank == 0:
            self.wq = ColumnParallelLinear(self.dim, self.n_heads * self.qk_head_dim)
        else:
            self.wq_a = Linear(self.dim, self.q_lora_rank)
            self.q_norm = RMSNorm(self.q_lora_rank)
            self.wq_b = ColumnParallelLinear(self.q_lora_rank, self.n_heads * self.qk_head_dim)
        self.wkv_a = Linear(self.dim, self.kv_lora_rank + self.qk_rope_head_dim)
        self.kv_norm = RMSNorm(self.kv_lora_rank)
        self.wkv_b = ColumnParallelLinear(self.kv_lora_rank, self.n_heads * (self.qk_nope_head_dim + self.v_head_dim))
        self.wo = RowParallelLinear(self.n_heads * self.v_head_dim, self.dim)
        self.softmax_scale = self.qk_head_dim ** -0.5
        if args.max_seq_len > args.original_seq_len:
            mscale = 0.1 * args.mscale * math.log(args.rope_factor) + 1.0
            self.softmax_scale = self.softmax_scale * mscale * mscale

        if attn_impl == "naive":
            self.register_buffer("k_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.qk_head_dim), persistent=False)
            self.register_buffer("v_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.n_local_heads, self.v_head_dim), persistent=False)
        else:
            self.register_buffer("kv_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.kv_lora_rank), persistent=False)
            self.register_buffer("pe_cache", torch.zeros(args.max_batch_size, args.max_seq_len, self.qk_rope_head_dim), persistent=False)

    def forward(self, x: torch.Tensor, start_pos: int, freqs_cis: torch.Tensor, mask: Optional[torch.Tensor]):
        """
        Forward pass for the Multi-Head Latent Attention (MLA) Layer.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, seq_len, dim).
            start_pos (int): Starting position in the sequence for caching.
            freqs_cis (torch.Tensor): Precomputed complex exponential values for rotary embeddings.
            mask (Optional[torch.Tensor]): Mask tensor to exclude certain positions from attention.

        Returns:
            torch.Tensor: Output tensor with the same shape as the input.
        """
        bsz, seqlen, _ = x.size()
        end_pos = start_pos + seqlen
        if self.q_lora_rank == 0:
            q = self.wq(x)
        else:
            q = self.wq_b(self.q_norm(self.wq_a(x)))
        q = q.view(bsz, seqlen, self.n_local_heads, self.qk_head_dim)
        q_nope, q_pe = torch.split(q, [self.qk_nope_head_dim, self.qk_rope_head_dim], dim=-1)
        q_pe = apply_rotary_emb(q_pe, freqs_cis)
        kv = self.wkv_a(x)
        kv, k_pe = torch.split(kv, [self.kv_lora_rank, self.qk_rope_head_dim], dim=-1)
        k_pe = apply_rotary_emb(k_pe.unsqueeze(2), freqs_cis)
        if attn_impl == "naive":
            q = torch.cat([q_nope, q_pe], dim=-1)
            kv = self.wkv_b(self.kv_norm(kv))
            kv = kv.view(bsz, seqlen, self.n_local_heads, self.qk_nope_head_dim + self.v_head_dim)
            k_nope, v = torch.split(kv, [self.qk_nope_head_dim, self.v_head_dim], dim=-1)
            k = torch.cat([k_nope, k_pe.expand(-1, -1, self.n_local_heads, -1)], dim=-1)
            self.k_cache[:bsz, start_pos:end_pos] = k
            self.v_cache[:bsz, start_pos:end_pos] = v
            scores = torch.einsum("bshd,bthd->bsht", q, self.k_cache[:bsz, :end_pos]) * self.softmax_scale
        else:
            wkv_b = self.wkv_b.weight if self.wkv_b.scale is None else weight_dequant(self.wkv_b.weight, self.wkv_b.scale, block_size) 
            wkv_b = wkv_b.view(self.n_local_heads, -1, self.kv_lora_rank)
            q_nope = torch.einsum("bshd,hdc->bshc", q_nope, wkv_b[:, :self.qk_nope_head_dim])
            self.kv_cache[:bsz, start_pos:end_pos] = self.kv_norm(kv)
            self.pe_cache[:bsz, start_pos:end_pos] = k_pe.squeeze(2)
            scores = (torch.einsum("bshc,btc->bsht", q_nope, self.kv_cache[:bsz, :end_pos]) +
                      torch.einsum("bshr,btr->bsht", q_pe, self.pe_cache[:bsz, :end_pos])) * self.softmax_scale
        if mask is not None:
            scores += mask.unsqueeze(1)
        scores = scores.softmax(dim=-1, dtype=torch.float32).type_as(x)
        if attn_impl == "naive":
            x = torch.einsum("bsht,bthd->bshd", scores, self.v_cache[:bsz, :end_pos])
        else:
            x = torch.einsum("bsht,btc->bshc", scores, self.kv_cache[:bsz, :end_pos])
            x = torch.einsum("bshc,hdc->bshd", x, wkv_b[:, -self.v_head_dim:])
        x = self.wo(x.flatten(2))
        return x
